# Banco de evaluación de neuronas, activaciones y capas

Compara las piezas de investigación de `02_capas_y_activaciones` (y la activación seno del 04) contra una red estándar del mismo tamaño:

- **mismas tareas** (regresión cúbica, círculos, lunas, XOR, serie seno y dígitos),
- **varias semillas** (5 por defecto),
- y un **veredicto honesto** (🟢 mejor, ⚪ igual, 🔴 peor), calculado con una prueba t pareada.

El código está en `laboratorio/banco/` (ver su README). Funciona en CPU y tarda unos 5 minutos.

## 1. Traer el repositorio

El repo es privado. Guarda un token de GitHub con permiso de solo lectura en **Secretos de Colab** (ícono de llave, a la izquierda) con el nombre `GITHUB_TOKEN`. **Nunca lo pegues en una celda.**

In [ ]:
import os, subprocess
from google.colab import userdata

if not os.path.exists("/content/Notbook"):
    token = userdata.get("GITHUB_TOKEN")
    subprocess.run(["git", "clone", "--depth", "1", f"https://{token}@github.com/SiriusMagic/Notbook.git", "/content/Notbook"], check=True)
%cd /content/Notbook
!pip install -q -r laboratorio/requirements.txt

## 2. Correr todo el banco

In [ ]:
!python -m laboratorio.banco --todo --semillas 5 --salida resultados

## 3. Ver el informe

In [ ]:
from IPython.display import Markdown, display
display(Markdown(open("resultados/informe.md", encoding="utf-8").read()))

## 4. Probar una sola pieza

Para trabajar en una idea nueva, agrégala a `PIEZAS` en `laboratorio/banco/piezas.py` y córrela solo en las tareas que te interesan:

In [ ]:
!python -m laboratorio.banco --pieza ida --pieza seno --tarea serie_seno --tarea lunas --semillas 5

## 5. Leer las corridas una por una

Cada entrenamiento queda en `resultados/AAAA-MM-DD.jsonl`, con la métrica en prueba y en entrenamiento, los parámetros y el tiempo.

In [ ]:
import pandas as pd, glob
corridas = pd.concat(pd.read_json(f, lines=True) for f in glob.glob("resultados/*.jsonl"))
corridas.pivot_table(index="pieza", columns="tarea", values="valor", aggfunc="mean").round(4)

## 6. Datos reales: imágenes y texto (Hugging Face)

Tareas disponibles: `mnist`, `fashion_mnist`, `cifar10` (imágenes) y `resenas_es` (reseñas en español, 1 a 5 estrellas), `ag_news`, `imdb` (texto).

- Se toman 10 000 ejemplos para entrenar y 2 000 de prueba, **al azar** (algunos conjuntos vienen ordenados por clase).
- **Parada temprana**: se separa un 15 % para validación y se corta si no mejora en 5 épocas; la red vuelve a su mejor época.
- **Guardado**: con `--guardar`, el mejor modelo de cada corrida queda en disco.

Para no descargar ni entrenar de nuevo cada vez que se reinicia Colab, guardamos todo en **Google Drive**. Si tienes un token de Hugging Face, ponlo en Secretos de Colab como `HF_TOKEN` (es opcional: solo da descargas más rápidas).

In [ ]:
import os
from google.colab import drive, userdata

drive.mount("/content/drive")
BASE = "/content/drive/MyDrive/banco"
os.environ["HF_HOME"] = f"{BASE}/huggingface"         # archivos originales de Hugging Face
os.environ["LABORATORIO_CACHE"] = f"{BASE}/tensores"  # datos ya preparados (carga en segundos)
try:
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
except Exception:
    pass  # sin token también funciona

Una prueba rápida (2 piezas, 2 tareas):

In [ ]:
!python -m laboratorio.banco --pieza ida --pieza seno --tarea mnist --tarea resenas_es --semillas 3 --guardar {BASE}/modelos --salida {BASE}/resultados

Todo el banco con datos reales (9 piezas × 6 tareas × 3 semillas; en CPU puede tardar más de una hora):

In [ ]:
!python -m laboratorio.banco --reales --guardar {BASE}/modelos --salida {BASE}/resultados

## 7. Recuperar un modelo guardado

Los modelos se guardan como `pieza__tarea__s<semilla>.pt` y se cargan con `weights_only=True` (solo pesos, nunca código).

In [ ]:
from laboratorio.banco.evaluar import cargar_modelo, _predecir, _medir
from laboratorio.banco.tareas import cargar_tarea

modelo = cargar_modelo(f"{BASE}/modelos/ida__mnist__s0.pt")
tarea = cargar_tarea("mnist")
print("exactitud en prueba:", _medir(tarea, _predecir(modelo, tarea.x_prueba), tarea.y_prueba))